In [0]:
dbutils.widgets.text("batch_id", "Batch_01")
dbutils.widgets.text(
    "source_path",
    "/Volumes/final_assesment/source/online_shopping_files/Batch_01_Online_Shopping.csv"
)

batch_id = dbutils.widgets.get("batch_id")
source_path = dbutils.widgets.get("source_path")

target_table = "final_assesment.bronze.orders_raw"

source_file = source_path.split("/")[-1]

print("Batch ID:", batch_id)
print("Source File:", source_file)
print("Target Table:", target_table)

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    IntegerType, DecimalType
)

from pyspark.sql.functions import (
    lit, current_timestamp, col
)

source_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_updated_at", TimestampType(), True),

    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("customer_dob", DateType(), True),

    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),

    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DecimalType(12, 2), True),
    StructField("discount", DecimalType(12, 2), True),

    StructField("payment_method_id", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("payment_provider", StringType(), True)
])

print("Source schema defined successfully.")

In [0]:
df_source = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(source_schema)
    .load(source_path)
)

source_count = df_source.count()

if source_count == 0:
    raise Exception("Source file is empty. Ingestion stopped.")

print("Source record count:", source_count)

display(df_source)

In [0]:
df_bronze = (
    df_source
    .withColumn("batch_id", lit(batch_id))
    .withColumn("source_file", lit(source_file))
    .withColumn("ingestion_timestamp", current_timestamp())
)

display(df_bronze)

In [0]:
if spark.catalog.tableExists(target_table):

    existing_count = (
        spark.table(target_table)
        .filter(col("batch_id") == batch_id)
        .count()
    )

    if existing_count == source_count:
        print(
            f"{batch_id} is already fully ingested. "
            "Skipping duplicate ingestion."
        )

    elif existing_count > 0:
        raise Exception(
            f"Partial ingestion detected for {batch_id}. "
            f"Existing: {existing_count}, "
            f"Expected: {source_count}. "
            "Investigate before retrying."
        )

    else:
        (
            df_bronze.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

        print(f"{batch_id} ingested successfully.")

else:

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print("Bronze table created and initial batch ingested.")

In [0]:
%sql
SELECT
    batch_id,
    source_file,
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS distinct_order_ids,
    MIN(ingestion_timestamp) AS first_ingestion,
    MAX(ingestion_timestamp) AS last_ingestion
FROM final_assesment.bronze.orders_raw
GROUP BY batch_id, source_file
ORDER BY batch_id;

In [0]:
%sql
SELECT COUNT(*) AS total_bronze_records
FROM final_assesment.bronze.orders_raw;